In [ ]:
"""
This script retrieves historical precipitation conditions for road traffic accidents using the
Open-Meteo Archive API. For each accident record, it searches backwards in time from the accident
timestamp to identify the most recent rainfall event, enabling the inclusion of antecedent weather
conditions within the accident dataset.

The workflow reads accident records containing geographic coordinates and timestamps, converts the
date and time fields into UTC datetime objects, and submits requests to the Open-Meteo Archive API
for hourly precipitation, rainfall, and snowfall observations covering the seven days preceding
each accident. API requests include automatic retry logic with exponential backoff and rate-limit
handling to improve reliability when processing large datasets.

Rather than using weather conditions only at the accident time, the script iteratively examines
hourly observations in six-hour intervals up to one week before each accident. The first historical
hour where both total precipitation and rainfall exceed zero is selected as the representative
antecedent rainfall event. If no qualifying rainfall is detected within the search window, the
record is flagged accordingly.

For every processed accident, the selected weather observation, search metadata, API response, and
processing status are stored in a JSON Lines output file. Records that encounter processing or API
errors are logged separately in a CSV file to facilitate validation and rerunning of failed cases.
The implementation also supports resuming interrupted executions by skipping accident records that
have already been successfully processed.

This automated pipeline provides a scalable approach for enriching large road accident datasets
with historical precipitation information, allowing antecedent weather conditions to be integrated
into subsequent road safety and accident severity analyses.
"""

In [ ]:
#!/usr/bin/env python3

import csv
import json
import os
import time
from datetime import datetime, timezone, timedelta

import pandas as pd
import requests
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================
INPUT_CSV = "all_59k_accidents_complete_final_2_cleaned.csv"
OUTPUT_JSONL = "open_meteo_archive_precipitation_backtracked.jsonl"
ERRORS_CSV = "open_meteo_errors.csv"

REQUEST_TIMEOUT = 30
MAX_RETRIES = 5
BACKOFF_FACTOR = 2.0
SLEEP_BETWEEN_REQUESTS = 0.0

# Hard stop for the run, so you stay under your monthly cap.
MONTHLY_CALL_BUDGET = int(os.getenv("OPEN_METEO_CALL_BUDGET", "2000000"))

# Set these via env if needed.
OPEN_METEO_API_KEY = os.getenv("OPEN_METEO_API_KEY", "").strip()
ARCHIVE_ENDPOINT = os.getenv(
    "OPEN_METEO_ARCHIVE_ENDPOINT",
    "https://archive-api.open-meteo.com/v1/archive",
)

# Open-Meteo controls
TIMEZONE = "UTC"
TIMEFORMAT = "iso8601"
CELL_SELECTION = "land"

# Use the provider’s combined default unless you want to override it.
ARCHIVE_MODELS = os.getenv("OPEN_METEO_ARCHIVE_MODELS", "best_match")

# If set, this overrides Open-Meteo’s default downscaling behavior.
# Leave blank to omit the parameter.
ELEVATION_OVERRIDE = os.getenv("OPEN_METEO_ELEVATION_OVERRIDE", "").strip()

session = requests.Session()
calls_made = 0

# Search settings
STEP_HOURS = 6
LOOKBACK_HOURS = 24 * 7  # 1 week

# CSV columns
OUTPUT_COLUMNS = [
    "_row_index",
    "accident_datetime_utc",
    "source_latitude",
    "source_longitude",
    "archive_status",
    "archive_error",
    "archive_request_count",
    "archive_selected_candidate_utc",
    "archive_selected_hours_back",
    "archive_selected_weather",
    "archive_json",
]


# ============================================================
# HELPERS
# ============================================================
def parse_accident_datetime_utc(row):
    raw_date = str(row.get("accidentdate", "")).strip()
    raw_time = str(row.get("time", "")).strip()

    if not raw_date:
        raise ValueError("missing accidentdate")
    if not raw_time:
        raise ValueError("missing time")

    try:
        date_part = datetime.strptime(raw_date, "%d-%b-%y")
    except Exception as exc:
        raise ValueError(f"invalid accidentdate format: {raw_date}") from exc

    try:
        time_part = datetime.strptime(raw_time, "%H:%M")
    except Exception as exc:
        raise ValueError(f"invalid time format: {raw_time}") from exc

    return datetime(
        date_part.year,
        date_part.month,
        date_part.day,
        time_part.hour,
        time_part.minute,
        tzinfo=timezone.utc,
    )


def get_base_params(lat, lon):
    params = {
        "latitude": float(lat),
        "longitude": float(lon),
        "timezone": TIMEZONE,
        "timeformat": TIMEFORMAT,
        "cell_selection": CELL_SELECTION,
        "models": ARCHIVE_MODELS,
    }

    if ELEVATION_OVERRIDE != "":
        params["elevation"] = ELEVATION_OVERRIDE

    if OPEN_METEO_API_KEY:
        params["apikey"] = OPEN_METEO_API_KEY

    return params


def request_json(endpoint, params):
    global calls_made

    attempt = 0
    wait = 2.0
    last_url = None

    while attempt < MAX_RETRIES:
        try:
            calls_made += 1
            resp = session.get(endpoint, params=params, timeout=REQUEST_TIMEOUT)
            last_url = resp.url

            if resp.status_code == 429:
                pause = 60 if attempt < 2 else 300
                print(f"\n[!] Rate limited (429). Sleeping {pause}s before retry {attempt + 1}...")
                time.sleep(pause)
                attempt += 1
                continue

            if resp.status_code == 400:
                return False, {
                    "error": f"400 bad request: {resp.text}",
                    "url": resp.url,
                }, resp.url

            resp.raise_for_status()
            return True, resp.json(), resp.url

        except Exception as exc:
            print(f"\n[!] Request error: {exc}. Retrying in {wait}s...")
            time.sleep(wait)
            wait *= BACKOFF_FACTOR
            attempt += 1

    return False, {"error": "Max retries exceeded", "url": last_url}, last_url


def parse_number_list(values):
    out = []
    for v in values or []:
        try:
            out.append(float(v))
        except Exception:
            out.append(None)
    return out


def build_time_index(times):
    """
    Map "YYYY-MM-DDTHH:MM" -> index in the returned hourly arrays.
    Open-Meteo hourly timestamps in UTC should match this format.
    """
    return {t: i for i, t in enumerate(times or [])}


def normalize_hour(dt):
    return dt.replace(minute=0, second=0, microsecond=0)


def fetch_archive_for_row(row_idx, lat, lon, accident_dt):
    """
    One request per accident:
      - fetch 7 days of hourly precipitation/rain/snowfall
      - scan backwards in 6-hour steps
      - stop when precipitation > 0 and rain > 0
      - stop after 1 week
    """
    base_params = get_base_params(lat, lon)

    start_date = (accident_dt - timedelta(days=7)).date().isoformat()
    end_date = accident_dt.date().isoformat()

    params = dict(base_params)
    params.update({
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "precipitation,rain,snowfall",
    })

    ok, payload_or_err, url = request_json(ARCHIVE_ENDPOINT, params)

    if not ok:
        return False, {
            "endpoint": ARCHIVE_ENDPOINT,
            "urls": [url],
            "requests": 1,
            "errors": [payload_or_err.get("error", "request failed")],
            "selected_candidate_utc": None,
            "selected_hours_back": None,
        }, {}

    payload = payload_or_err
    hourly = payload.get("hourly", {})

    times = hourly.get("time", [])
    precipitation_arr = parse_number_list(hourly.get("precipitation", []))
    rain_arr = parse_number_list(hourly.get("rain", []))
    snowfall_arr = parse_number_list(hourly.get("snowfall", []))

    if not times:
        return False, {
            "endpoint": ARCHIVE_ENDPOINT,
            "urls": [url],
            "requests": 1,
            "errors": ["missing hourly.time"],
            "selected_candidate_utc": None,
            "selected_hours_back": None,
        }, payload

    time_to_idx = build_time_index(times)

    selected_weather = None
    selected_candidate_utc = None
    selected_hours_back = None

    for hours_back in range(STEP_HOURS, LOOKBACK_HOURS + 1, STEP_HOURS):
        candidate_dt = accident_dt - timedelta(hours=hours_back)
        candidate_hour = normalize_hour(candidate_dt).strftime("%Y-%m-%dT%H:%M")

        idx = time_to_idx.get(candidate_hour)
        if idx is None:
            continue

        precipitation = precipitation_arr[idx] if idx < len(precipitation_arr) else None
        rain = rain_arr[idx] if idx < len(rain_arr) else None
        snowfall = snowfall_arr[idx] if idx < len(snowfall_arr) else None

        if precipitation is None or rain is None:
            continue

        if precipitation > 0 and rain > 0:
            selected_weather = {
                "timestamp_utc": candidate_hour,
                "hours_back": hours_back,
                "precipitation": precipitation,
                "rain": rain,
                "snowfall": snowfall,
            }
            selected_candidate_utc = candidate_hour
            selected_hours_back = hours_back
            break

    errors = []
    if selected_weather is None:
        errors.append("no qualifying hourly observation found within 7 days")

    payload["_selected_weather"] = selected_weather

    archive_meta = {
        "endpoint": ARCHIVE_ENDPOINT,
        "urls": [url],
        "requests": 1,
        "errors": errors,
        "selected_candidate_utc": selected_candidate_utc,
        "selected_hours_back": selected_hours_back,
    }

    return selected_weather is not None, archive_meta, payload


def empty_payload():
    return {}


# ============================================================
# MAIN LOOP
# ============================================================
def run_fetcher():
    df = pd.read_csv(INPUT_CSV, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    completed_rows = set()

    if os.path.exists(OUTPUT_JSONL):
        with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    completed_rows.add(json.loads(line)["_row_index"])
                except Exception:
                    pass

    if not os.path.exists(OUTPUT_JSONL):
        open(OUTPUT_JSONL, "a").close()

    if os.path.exists(ERRORS_CSV):
        os.remove(ERRORS_CSV)

    errors_header_written = False
    success_rows = 0
    error_rows = 0

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Fetching accident rows"):
        if idx in completed_rows:
            continue

        if calls_made >= MONTHLY_CALL_BUDGET:
            print(
                f"\n[!] Monthly call budget reached "
                f"({calls_made} >= {MONTHLY_CALL_BUDGET}). Stopping."
            )
            break

        try:
            row = row.fillna("")
            lat_raw = row.get("latitude", row.get("source_latitude"))
            lon_raw = row.get("longitude", row.get("source_longitude"))

            if (
                pd.isna(lat_raw)
                or pd.isna(lon_raw)
                or str(lat_raw).strip() == ""
                or str(lon_raw).strip() == ""
            ):
                raise ValueError("missing lat/lon")

            lat = float(str(lat_raw).strip())
            lon = float(str(lon_raw).strip())
            accident_dt = parse_accident_datetime_utc(row)

            archive_ok, archive_meta, archive_payload = fetch_archive_for_row(
                idx, lat, lon, accident_dt
            )

            out_row = {
                "_row_index": idx,
                "accident_datetime_utc": accident_dt.isoformat(),
                "source_latitude": lat,
                "source_longitude": lon,
                "archive_status": "ok" if archive_ok else "partial",
                "archive_error": archive_meta["errors"],
                "archive_request_count": archive_meta["requests"],
                "archive_selected_candidate_utc": archive_meta["selected_candidate_utc"],
                "archive_selected_hours_back": archive_meta["selected_hours_back"],
                "archive_selected_weather": archive_payload.get("_selected_weather"),
                "archive_json": archive_payload,
            }

            with open(OUTPUT_JSONL, "a", encoding="utf-8") as f:
                f.write(json.dumps(out_row, ensure_ascii=False) + "\n")

            success_rows += 1

            if not archive_ok:
                with open(ERRORS_CSV, "a", newline="", encoding="utf-8") as ef:
                    writer = csv.writer(ef)
                    if not errors_header_written:
                        writer.writerow(["row_index", "type", "message"])
                        errors_header_written = True
                    writer.writerow([
                        idx,
                        "partial",
                        f"archive={archive_meta['errors']}",
                    ])

        except Exception as exc:
            error_rows += 1
            with open(ERRORS_CSV, "a", newline="", encoding="utf-8") as ef:
                writer = csv.writer(ef)
                if not errors_header_written:
                    writer.writerow(["row_index", "type", "message"])
                    errors_header_written = True
                writer.writerow([idx, "fatal", str(exc)])

        if SLEEP_BETWEEN_REQUESTS:
            time.sleep(SLEEP_BETWEEN_REQUESTS)

    print(
        f"\nDone. Rows saved: {success_rows}, errors: {error_rows}, "
        f"HTTP calls made: {calls_made}"
    )


if __name__ == "__main__":
    if not os.path.exists(INPUT_CSV):
        print(f"Error: {INPUT_CSV} not found in current directory.")
        raise SystemExit(1)

    run_fetcher()